# FallNet StaticLMU — Accuracy Verification
Loads the best trained `.keras` fold and verifies accuracy + Fall_Initiation recall on the full dataset.

In [ ]:
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    recall_score, f1_score, accuracy_score
)

import tensorflow as tf
from tensorflow import keras

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Paths

In [ ]:
base_dir    = Path('~/repos/summerschool2023/projects/fall-detection/fall_detection_data').expanduser()
processed_dir = base_dir / 'processed'
models_dir  = base_dir / 'models'

print(f'Data dir:   {processed_dir}')
print(f'Models dir: {models_dir}')

# Show available trained keras folds
keras_folds = sorted(models_dir.glob('fallnet_hybrid_staticlmu_fold_*.keras'))
print(f'\nFound {len(keras_folds)} trained fold(s):')
for f in keras_folds:
    print(f'  {f.name}')

## 2. Load Data

In [ ]:
X_data   = np.load(processed_dir / 'X_data_6class.npy')
y_labels = np.load(processed_dir / 'y_labels_6class.npy')

with open(processed_dir / 'label_map_6class.json') as f:
    label_map = json.load(f)
reverse_label_map = {v: k for k, v in label_map.items()}

print(f'X_data:   {X_data.shape}   dtype={X_data.dtype}')
print(f'y_labels: {y_labels.shape}  classes={sorted(np.unique(y_labels))}')
print()
from collections import Counter
counts = Counter(y_labels)
print('Class distribution:')
for idx in sorted(counts):
    print(f'  [{idx}] {reverse_label_map[idx]:<35s}: {counts[idx]:5d} ({counts[idx]/len(y_labels)*100:.1f}%)')

## 3. Define StaticLMU (required to load .keras)

In [ ]:
from tensorflow.keras import layers
from scipy.linalg import expm

@keras.utils.register_keras_serializable()
class StaticLMUCell(layers.Layer):
    def __init__(self, input_dim, memory_d, order, theta, hidden_units, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.input_dim    = input_dim
        self.memory_d     = memory_d
        self.order        = order
        self.theta        = theta
        self.hidden_units = hidden_units
        self.dropout_rate = dropout
        self.memory_size  = memory_d * order

    def build(self, input_shape):
        A, B = self._get_legendre_matrices(self.order, self.theta)
        self.A = self.add_weight('A', shape=(self.order, self.order),
                                 initializer=keras.initializers.Constant(A), trainable=False)
        self.B = self.add_weight('B', shape=(self.order, 1),
                                 initializer=keras.initializers.Constant(B), trainable=False)
        self.encoder       = layers.Dense(self.memory_d, kernel_regularizer=keras.regularizers.l2(1e-5), name='lmu_encoder')
        self.hidden_dense1 = layers.Dense(self.hidden_units, activation='tanh', kernel_regularizer=keras.regularizers.l2(1e-5), name='lmu_hidden1')
        self.hidden_dense2 = layers.Dense(self.hidden_units, activation='tanh', kernel_regularizer=keras.regularizers.l2(1e-5), name='lmu_hidden2')
        if self.dropout_rate > 0:
            self.dropout = layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    @staticmethod
    def _get_legendre_matrices(order, theta):
        Q = np.arange(order, dtype=np.float64)
        R = (2*Q+1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A_cont = np.where(i < j, -1, (-1.0)**(i-j+1)) * R
        B_cont = (-1.0)**Q[:, None] * R
        A_d = expm(A_cont / theta)
        B_d = np.linalg.solve(A_cont, (A_d - np.eye(order)) @ B_cont)
        return A_d.astype(np.float32), B_d.astype(np.float32)

    def call(self, x_t, memory_state, training=False):
        u_t = self.encoder(x_t)
        Am  = tf.matmul(memory_state, self.A, transpose_b=True)
        Bu  = tf.expand_dims(u_t, axis=-1) * tf.reshape(self.B, [1, 1, self.order])
        new_memory = Am + Bu
        m_flat  = tf.reshape(new_memory, [-1, self.memory_size])
        h_input = tf.concat([x_t, m_flat], axis=-1)
        h_t = self.hidden_dense1(h_input)
        if self.dropout_rate > 0:
            h_t = self.dropout(h_t, training=training)
        h_t = self.hidden_dense2(h_t)
        return h_t, new_memory

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(input_dim=self.input_dim, memory_d=self.memory_d, order=self.order,
                        theta=self.theta, hidden_units=self.hidden_units, dropout=self.dropout_rate))
        return cfg


@keras.utils.register_keras_serializable()
class StaticLMU(layers.Layer):
    def __init__(self, memory_d, order, theta, hidden_units, dropout=0.0, return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.memory_d        = memory_d
        self.order           = order
        self.theta           = theta
        self.hidden_units    = hidden_units
        self.dropout_rate    = dropout
        self.return_sequences = return_sequences

    def build(self, input_shape):
        self.timesteps = input_shape[1]
        self.input_dim = input_shape[2]
        self.cell = StaticLMUCell(input_dim=self.input_dim, memory_d=self.memory_d, order=self.order,
                                  theta=self.theta, hidden_units=self.hidden_units,
                                  dropout=self.dropout_rate, name='lmu_cell')
        super().build(input_shape)

    def call(self, inputs, training=False):
        batch_size = tf.shape(inputs)[0]
        memory  = tf.zeros([batch_size, self.memory_d, self.order])
        x_steps = tf.unstack(inputs, num=self.timesteps, axis=1)
        outputs = []
        for t in range(self.timesteps):
            h_t, memory = self.cell(x_steps[t], memory, training=training)
            if self.return_sequences:
                outputs.append(h_t)
        return tf.stack(outputs, axis=1) if self.return_sequences else h_t

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(memory_d=self.memory_d, order=self.order, theta=self.theta,
                        hidden_units=self.hidden_units, dropout=self.dropout_rate,
                        return_sequences=self.return_sequences))
        return cfg


print('✅ StaticLMUCell + StaticLMU registered')

## 4. Evaluate All Folds, Pick Best

In [ ]:
fall_init_idx = label_map['Fall_Initiation']
fold_results  = []

for fold_path in keras_folds:
    print(f'Loading {fold_path.name}...')
    model = keras.models.load_model(fold_path)
    y_pred_probs = model.predict(X_data, verbose=0, batch_size=256)
    y_pred = np.argmax(y_pred_probs, axis=1)

    acc        = accuracy_score(y_labels, y_pred)
    fi_recall  = recall_score(y_labels == fall_init_idx, y_pred == fall_init_idx)
    fi_f1      = f1_score(y_labels == fall_init_idx, y_pred == fall_init_idx)

    fold_results.append({
        'path': fold_path, 'model': model,
        'y_pred': y_pred,
        'accuracy': acc, 'fi_recall': fi_recall, 'fi_f1': fi_f1
    })
    print(f'  Accuracy={acc:.4f}  FI_Recall={fi_recall:.4f}  FI_F1={fi_f1:.4f}')

best = max(fold_results, key=lambda r: r['fi_f1'])
print(f'\n✅ Best fold: {best["path"].name}')
print(f'   Accuracy:          {best["accuracy"]:.4f}')
print(f'   FI Recall:         {best["fi_recall"]:.4f}')
print(f'   FI F1:             {best["fi_f1"]:.4f}')

In [ ]:
keras_folds = sorted(models_dir.glob('fallnet_hybrid_fold_*.keras'))

## 5. Full Classification Report

In [ ]:
class_names = [reverse_label_map[i] for i in range(6)]
y_pred_best = best['y_pred']

print('=' * 80)
print('CLASSIFICATION REPORT — Best Fold')
print('=' * 80)
print(classification_report(y_labels, y_pred_best, target_names=class_names, digits=4))

# Per-class Fall_Init focus
print('\nFall_Initiation detail:')
fi_mask = y_labels == fall_init_idx
print(f'  True positives:  {np.sum((y_pred_best == fall_init_idx) & fi_mask)}')
print(f'  False negatives: {np.sum((y_pred_best != fall_init_idx) & fi_mask)}')
print(f'  Total FI samples:{np.sum(fi_mask)}')

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_labels, y_pred_best)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title(f'FallNet StaticLMU — Confusion Matrix\n'
             f'Accuracy={best["accuracy"]:.4f}  FI_Recall={best["fi_recall"]:.4f}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig(models_dir / 'verify_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: verify_confusion_matrix.png')

## 7. Per-Class Accuracy Bar Chart

In [ ]:
per_class_recall = cm.diagonal() / cm.sum(axis=1)

colors = ['#4CAF50' if r >= 0.90 else '#FF9800' if r >= 0.75 else '#F44336'
          for r in per_class_recall]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(class_names, per_class_recall * 100, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(90, color='green', linestyle='--', alpha=0.6, label='90% threshold')
ax.axhline(75, color='orange', linestyle='--', alpha=0.6, label='75% threshold')
for bar, val in zip(bars, per_class_recall):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Recall (%)')
ax.set_title('Per-Class Recall — FallNet StaticLMU', fontweight='bold')
ax.set_ylim(0, 110)
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(models_dir / 'verify_per_class_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: verify_per_class_recall.png')

## 8. Summary

In [ ]:
print('=' * 60)
print('VERIFICATION SUMMARY')
print('=' * 60)
print(f'Model:             {best["path"].name}')
print(f'Dataset:           {len(y_labels):,} samples, 6 classes')
print(f'Overall Accuracy:  {best["accuracy"]:.4f} ({best["accuracy"]*100:.2f}%)')
print(f'FI Recall:         {best["fi_recall"]:.4f}')
print(f'FI F1:             {best["fi_f1"]:.4f}')
print()
print('Per-class recall:')
for i, (name, r) in enumerate(zip(class_names, per_class_recall)):
    flag = '⚠️' if r < 0.75 else ''
    print(f'  [{i}] {name:<35s}: {r*100:.1f}% {flag}')
print()
deploy_ready = best['fi_recall'] >= 0.95
print(f'Deploy ready (FI recall ≥ 95%): {"✅ YES" if deploy_ready else "❌ NO — check FI recall"}')